In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 171 nodes, deleted 253 relationships, completed after 569 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 5405 ms.


### Node Rules 

In [8]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env.json")
# env = Environment("../dtgraph/type_checking/env_common_movies.json")


generate_humans = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(:Movie)
GENERATE
(x = (p):Actor {
    name = p.name,
    age = p.born + "123"
})
''',
env=env,
type_strict = True)

# common_movies = Rule('''
# MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)
# WHERE id(x) < id(z)
# GENERATE
# ((x):Actor {
#     name = x.name
# })-[(x,z):ACTED_WITH {
#     CommonMovies = [y.title]
# }]->((z):Actor {
#     name = z.name
# })
# ''',
# env=env,
# type_strict = True
# )


# generate_created = Rule('''
# MATCH (p:Person)-[:DIRECTED]->(m:Movie)
# GENERATE
# (x = (p):)-[():CREATED {
#     role = 123
# }]->(y = (m):)
# ''',
# env=env,
# type_strict = True)

### Execute Rules

In [9]:
my_transform = Transformation([generate_humans])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 13 ms.
AST:
PropertyAccess
    ├── var: p
    └── prop: name
AST:
BinaryExpression: +
    └── PropertyAccess
        ├── var: p
        └── prop: born
    └── Literal
        ├── value: 123
        └── type: string
Rule: Added 204 labels, created 102 nodes, set 446 properties, created 0 relationships, completed after 1733 ms.


1733

### Abort Transformation

In [10]:
my_transform.abort()

Index: Removed 1 index, completed after 76 ms.
Abort: Deleted 102 nodes, deleted 0 relationships, completed after 171 ms.
